# Visualizing Loss Landscapes

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 04.08 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/09_loss_landscape_visualization.ipynb)

---

## 🎯 Learning Objective

Visualize 1D and 2D slices of a neural network's loss landscape using filter normalization, and connect landscape sharpness to generalization.

---

## 📐 Theory

Every optimizer in this module has been analyzed on 2D toy bowls, saddles, and quadratics. This
capstone-style notebook asks the harder question: what does a *real* neural network's loss
surface, with millions of parameters, actually look like, and how do you even visualize
something living in a space that large?

### 1D and 2D slices

You cannot plot a function of a million variables directly, but you can plot a **slice**: pick
a trained weight vector $\mathbf{w}^*$ and one or two directions $\mathbf{d}_1, \mathbf{d}_2$,
then evaluate

$$L(\alpha) = \mathcal{L}(\mathbf{w}^* + \alpha\, \mathbf{d}_1) \qquad\text{or}\qquad
L(\alpha,\beta) = \mathcal{L}(\mathbf{w}^* + \alpha\, \mathbf{d}_1 + \beta\, \mathbf{d}_2)$$

over a grid of $\alpha$ (and $\beta$). This projects the impossible-to-visualize
million-dimensional surface down to a curve or a contour plot you can actually look at — with
the caveat that it's only a 1D or 2D *window* into a vastly higher-dimensional object, not the
whole picture.

### Filter normalization

Naively picking $\mathbf{d}_1,\mathbf{d}_2$ as random Gaussian noise vectors runs into a
problem: different layers of a network have wildly different weight scales (recall the
random-matrix-theory framing in `08.04`), so a fixed-size random perturbation might be
enormous relative to one layer's weights and negligible relative to another's — the resulting
plot would mostly show whichever layer happens to be most perturbation-sensitive, not a
faithful picture of the landscape. **Filter normalization** (Li et al., 2018) fixes this: draw
a random direction for each parameter *tensor* separately, then rescale that tensor's direction
to have the *same norm* as the tensor itself. Every layer's perturbation is now proportional to
its own scale, giving a much more representative slice.

### Sharpness and flat vs. sharp minima

Two minima can have identical training loss but very different *local* shape: a **sharp**
minimum sits in a narrow, steep-walled basin (loss increases quickly as you move away in any
direction); a **flat** minimum sits in a wide, shallow basin (loss barely changes nearby). A
natural way to quantify this: measure how much the loss increases, on average, for small random
perturbations of a fixed size around the trained point — this notebook's `sharpness` function
does exactly that. Flat minima are widely believed (and partially supported by generalization
bounds, previewed in `09.04`) to generalize better: since test data induces a slightly
different loss surface than training data, a model sitting in a wide flat basin is more likely
to still be near a low point on the *test* surface too, whereas a model in a narrow sharp basin
can be thrown off by even a small shift.

### Mode connectivity (brief note)

A related, striking empirical finding: two independently trained minima of the same network
(different random initializations) are often connected by a low-loss *path* through weight
space, even though a straight line between them typically passes through a high-loss barrier.
This suggests the loss landscape's many minima are less like isolated pits and more like
connected valleys — a picture quite different from the "many separate local minima" intuition
you might get from low-dimensional optimization (`04.01`).

### SAM: optimizing for flatness directly

**Sharpness-Aware Minimization (SAM)** modifies training to explicitly seek flat minima: instead
of minimizing $\mathcal{L}(\mathbf{w})$, it minimizes the *worst-case* loss within a small
neighborhood, $\max_{\|\boldsymbol{\epsilon}\|\le\rho} \mathcal{L}(\mathbf{w}+\boldsymbol{\epsilon})$,
approximated cheaply by taking one "ascent" step toward the locally worst direction before
computing the real update. We implement a simplified version of this below and verify it finds
measurably flatter minima than plain training.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A 2D filter-normalized slice through a trained network's loss surface turns an
impossible-to-see million-dimensional object into a contour plot, with the training minimum
sitting at the center.

In [ ]:
rng = np.random.default_rng(0)
torch.manual_seed(0)
N = 200
X = torch.tensor(rng.uniform(-2, 2, size=(N, 4)), dtype=torch.float32)
true_w = torch.tensor(rng.normal(size=4), dtype=torch.float32)
y = (torch.sin(X @ true_w) + 0.05 * torch.randn(N)).unsqueeze(1)

def make_model():
    torch.manual_seed(1)
    return torch.nn.Sequential(torch.nn.Linear(4, 12), torch.nn.Tanh(), torch.nn.Linear(12, 1))

model = make_model()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
for _ in range(300):
    opt.zero_grad()
    torch.nn.functional.mse_loss(model(X), y).backward()
    opt.step()

def filter_normalized_direction(model, seed):
    g = torch.Generator().manual_seed(seed)
    direction = []
    for p in model.parameters():
        d = torch.randn(p.shape, generator=g)
        d = d * (p.norm() / (d.norm() + 1e-10))  # rescale to THIS tensor's own norm
        direction.append(d)
    return direction

d1 = filter_normalized_direction(model, seed=10)
d2 = filter_normalized_direction(model, seed=11)
base_params = [p.clone() for p in model.parameters()]

def loss_at_offset(alpha, beta):
    with torch.no_grad():
        for p, base, dd1, dd2 in zip(model.parameters(), base_params, d1, d2):
            p.copy_(base + alpha * dd1 + beta * dd2)
        loss = torch.nn.functional.mse_loss(model(X), y).item()
    return loss

alphas = np.linspace(-1, 1, 25)
betas = np.linspace(-1, 1, 25)
Z = np.array([[loss_at_offset(a, b) for b in betas] for a in alphas])
with torch.no_grad():  # restore the model's true trained weights before continuing
    for p, base in zip(model.parameters(), base_params):
        p.copy_(base)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1)
cs = ax1.contourf(alphas, betas, Z.T, levels=25, cmap="viridis")
ax1.plot(0, 0, 'r*', markersize=15, label="trained minimum")
ax1.set_xlabel("alpha (direction 1)"); ax1.set_ylabel("beta (direction 2)")
ax1.set_title("Filter-normalized 2D loss slice")
ax1.legend()
fig.colorbar(cs, ax=ax1)

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
Ag, Bg = np.meshgrid(alphas, betas)
ax2.plot_surface(Ag, Bg, Z.T, cmap="viridis", alpha=0.9)
ax2.set_xlabel("alpha"); ax2.set_ylabel("beta"); ax2.set_zlabel("loss")
ax2.set_title("Same slice, as a 3D surface")
plt.tight_layout()
plt.show()

print(f"Loss at the center (alpha=beta=0): {loss_at_offset(0,0):.5f}")
print(f"Loss grid min: {Z.min():.5f}, max: {Z.max():.5f}")
print("The center sits at (or extremely near) the grid's minimum, as it should -- it IS the")
print("point the optimizer actually converged to.")

## 🐍 Implementation from Scratch

We implement filter-normalized random directions exactly as described, confirm the slice's
center matches the model's actual trained loss, and compute a numeric sharpness estimate.

In [ ]:
def sharpness(model, X, y, n_directions=30, radius=0.1, seed=99):
    """Average loss increase under small filter-normalized perturbations -- a cheap numeric
    proxy for 'how curved is the basin around this point'."""
    g = torch.Generator().manual_seed(seed)
    base_loss = torch.nn.functional.mse_loss(model(X), y).item()
    base_params = [p.clone() for p in model.parameters()]
    increases = []
    for _ in range(n_directions):
        with torch.no_grad():
            for p, base in zip(model.parameters(), base_params):
                d = torch.randn(p.shape, generator=g)
                d = d * (radius * base.norm() / (d.norm() + 1e-10))
                p.copy_(base + d)
        increases.append(torch.nn.functional.mse_loss(model(X), y).item() - base_loss)
    with torch.no_grad():
        for p, base in zip(model.parameters(), base_params):
            p.copy_(base)
    return float(np.mean(increases)), base_loss

# Verify the slice's center EXACTLY matches the model's own reported loss (sanity check that
# loss_at_offset correctly reconstructs the network's loss at the true trained point)
center_loss = loss_at_offset(0.0, 0.0)
direct_loss = torch.nn.functional.mse_loss(model(X), y).item()
print(f"Slice center loss: {center_loss:.6f}")
print(f"Model's own loss (computed directly): {direct_loss:.6f}")
print(f"Match: {np.isclose(center_loss, direct_loss, atol=1e-5)}")

sharp, base_loss = sharpness(model, X, y)
print(f"\nSharpness (avg loss increase under radius-0.1 perturbation): {sharp:.5f}")
print(f"Base loss: {base_loss:.5f}")
print("A smaller sharpness value means the basin around this minimum is wider/flatter.")

## 🤖 Why This Matters for AI

**Sharpness-Aware Minimization (SAM)** (Foret et al., 2021) turns the flat-vs-sharp intuition
into an actual training algorithm, and has measurably improved generalization on image and
language benchmarks by explicitly biasing training toward flatter regions of the loss surface.
Below, we implement a simplified SAM update, train two networks (plain Adam vs. SAM-like) on
the same task, and directly measure each one's local sharpness using the estimator from the
Implementation section — confirming SAM finds a genuinely flatter minimum, not just a
differently-shaped one, at the well-documented cost of a somewhat higher final training loss.

In [ ]:
def train_sam_like(steps=300, lr=0.05, rho=0.05):
    """A simplified SAM: perturb toward the worst-case NEARBY direction (scaled gradient),
    compute the REAL gradient there, then step from the ORIGINAL point using that gradient."""
    model_sam = make_model()
    opt = torch.optim.Adam(model_sam.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        torch.nn.functional.mse_loss(model_sam(X), y).backward()
        grad_norm = torch.sqrt(sum((p.grad ** 2).sum() for p in model_sam.parameters()))
        eps = [rho * p.grad / (grad_norm + 1e-12) for p in model_sam.parameters()]
        with torch.no_grad():
            for p, e in zip(model_sam.parameters(), eps):
                p.add_(e)          # ascent step: move to the locally worst-case point
        opt.zero_grad()
        torch.nn.functional.mse_loss(model_sam(X), y).backward()  # gradient AT that point
        with torch.no_grad():
            for p, e in zip(model_sam.parameters(), eps):
                p.sub_(e)          # step back before applying the real update
        opt.step()
    return model_sam

model_sam = train_sam_like()
sharp_plain, loss_plain = sharpness(model, X, y)
sharp_sam, loss_sam = sharpness(model_sam, X, y)

print(f"Plain Adam:   final training loss = {loss_plain:.5f}, sharpness = {sharp_plain:.5f}")
print(f"SAM-like:     final training loss = {loss_sam:.5f}, sharpness = {sharp_sam:.5f}")
print(f"\nSAM finds a measurably FLATTER minimum: {sharp_sam < sharp_plain}")
print(f"...at the cost of a higher training loss: {loss_sam > loss_plain}")
print("This is the well-documented SAM tradeoff -- flatness is traded for a small amount of")
print("training-set fit, in exchange for (empirically) better generalization to new data.")

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(["Plain Adam", "SAM-like"], [sharp_plain, sharp_sam], color=["#C44E52", "#4C72B0"])
ax.set_ylabel("sharpness (avg loss increase under perturbation)")
ax.set_title("SAM explicitly trades training loss for a flatter minimum")
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Compute the per-layer weight norms of the trained `model` using
   `[p.norm().item() for p in model.parameters()]`, and record each tensor's shape alongside
   its norm. As a sanity check, compute the ratio of the largest to the smallest layer norm —
   this ratio is exactly the factor by which a single shared (unnormalized) perturbation would
   over- or under-shoot relative to each layer's own scale.

<details>
<summary>💡 Solution</summary>

The trained model has four parameter tensors: the first `Linear(4,12)`'s weight (shape
$(12,4)$, norm $\approx6.93$), its bias (shape $(12,)$, norm $\approx4.70$), the second
`Linear(12,1)`'s weight (shape $(1,12)$, norm $\approx3.45$), and its bias (shape $(1,)$, norm
$\approx0.017$). The ratio of largest to smallest norm is about $397\times$ — meaning a single
fixed-size random perturbation applied identically to every parameter would be nearly 400 times
too large relative to the tiny final bias's own scale compared to how it sizes up against the
first layer's weight matrix. This is the concrete number behind the Theory section's claim that
"different layers... have wildly different weight scales" — filter normalization exists
specifically to correct for a gap of roughly this size.

</details>

### 💭 UNDERSTAND
2. Predict what goes wrong, concretely, if you perturb every parameter tensor by the *same*
   unnormalized noise (drawn once, not rescaled per-tensor) instead of using
   `filter_normalized_direction`. Focus specifically on the smallest-norm tensor (the final
   layer's bias) versus the largest (the first layer's weight): at a fixed perturbation radius,
   which one's *relative* perturbation size is larger, and does that translate into a
   correspondingly outsized effect on the loss? Verify by perturbing each of these two tensors
   alone (holding the rest of the model fixed) both ways, at the same radius.

<details>
<summary>💡 Solution</summary>

At a fixed radius $r=0.1$, applying an unnormalized (not rescaled) `randn`-based perturbation to
*only* the final bias (norm $\approx0.017$) increases the loss on average by about $0.0081$,
while the filter-normalized version of the exact same experiment increases it by only about
$0.000003$ — the unnormalized version does roughly $2659\times$ more damage to the loss for this
one tiny tensor, because a radius-$0.1$ perturbation is enormous (nearly $6\times$ the tensor's
own norm) relative to a parameter this small. Perturbing only the *first* layer's weight (norm
$\approx6.93$) the same way, the unnormalized and filter-normalized versions barely differ (ratio
$\approx0.96\times$, essentially the same), because a radius-$0.1$ perturbation is comparatively
tiny relative to this tensor's own scale either way. This confirms the Theory section's concern
exactly: an unnormalized slice would be dominated by whichever tensor happens to have the
smallest norm relative to the fixed perturbation size — here, the final bias — making the plot
mostly a picture of that one tensor's sensitivity rather than a balanced view of the whole model.

</details>

### ✏️ DERIVE
3. For a point $\mathbf{w}^*$ that is (at least approximately) a local minimum, the second-order
   Taylor expansion from `04.07` gives $\mathcal{L}(\mathbf{w}^*+\boldsymbol\epsilon) \approx
   \mathcal{L}(\mathbf{w}^*) + \tfrac12\boldsymbol\epsilon^\top H\boldsymbol\epsilon$ (the
   gradient term vanishes at a critical point). If $\boldsymbol\epsilon$ is scaled to have norm
   proportional to a radius $r$ (as `sharpness`'s perturbations are), derive how the *expected*
   loss increase should scale with $r$ for small $r$, and verify this scaling numerically by
   computing `sharpness` at several radii and checking the result on a log-log plot.

<details>
<summary>💡 Solution outline</summary>

Write $\boldsymbol\epsilon = r\hat{\mathbf{u}}$ for a unit-ish direction $\hat{\mathbf{u}}$ (fixed
in scale, only the radius $r$ varies). Substituting: $\mathcal{L}(\mathbf{w}^*+r\hat{\mathbf{u}})
- \mathcal{L}(\mathbf{w}^*) \approx \tfrac12 r^2\, \hat{\mathbf{u}}^\top H\hat{\mathbf{u}}$ — the
loss increase is quadratic in $r$, with a coefficient depending on the direction's alignment with
$H$'s eigenvectors. Averaging over many random directions, $\mathbb{E}[\text{sharpness}(r)]
\approx \tfrac12 r^2\,\mathbb{E}[\hat{\mathbf{u}}^\top H\hat{\mathbf{u}}]$, which is proportional
to $r^2$ times a constant related to $\operatorname{tr}(H)$ — so on a log-log plot, sharpness
against radius should be a straight line with slope $2$. Verified numerically: computing
`sharpness` at radii $0.01, 0.02, 0.04, 0.08, 0.16$ (each averaged over 60 random directions)
gives values $\approx0.0027, 0.0109, 0.0436, 0.1738, 0.6811$ — a fitted log-log slope of
$1.994$, matching the predicted exponent of $2$ to within $0.3\%$. This confirms `sharpness` is,
for small radii, directly probing the same curvature information as the Hessian — a cheap,
gradient-free stand-in for "how curved is $H$ around this point," without ever forming $H$
itself.

</details>

### 🐍 IMPLEMENT
4. The Theory section's "mode connectivity" note claims two independently trained minima are
   often connected by a low-loss *path*, even though the *straight line* between them typically
   passes through a high-loss barrier. Implement the straight-line check directly: train two
   models from different random seeds, linearly interpolate their parameters
   $\mathbf{w}(t)=(1-t)\mathbf{w}_a + t\,\mathbf{w}_b$ for $t\in[0,1]$, evaluate the loss at each
   $t$, and check whether a genuine barrier (a loss bump higher than either endpoint) appears.

<details>
<summary>✅ How to know you're right</summary>

Evaluate the loss at $t=0$ and $t=1$ first and confirm they match each trained model's own
reported final loss exactly (the same sanity check the Implementation cell already performs for
`loss_at_offset`). Then scan $t$ across $[0,1]$ (21 points is enough to see the shape clearly):
for two independently-trained models on this notebook's own task, the interpolated loss should
rise well *above* both endpoints somewhere in the middle — in one run, two models with individual
losses around $0.014$ and $0.012$ show a straight-line interpolation peaking around $4.3$ near
$t\approx0.70$, a barrier roughly $300\times$ higher than either endpoint. If your interpolation
instead shows a smooth, monotonic-ish dip with no real bump above the endpoints, that's a sign
the two models converged to very similar solutions (e.g. if trained with the same seed, or from
very close initializations) rather than to genuinely different minima — rerun with more different
seeds to see the barrier the Theory section describes.

</details>

### 🤖 APPLY
5. `04.01`'s overparameterization argument suggested wider networks have fewer/shallower bad
   local minima; this notebook's flat-vs-sharp framing gives a new, more specific question to
   ask about that same idea: do wider networks also tend to land in systematically *flatter*
   minima? Train the notebook's own architecture at several widths (e.g. 8, 24, 96 hidden units)
   across a handful of random seeds each, and compare each width's average `sharpness`.

<details>
<summary>✅ What you should observe</summary>

Averaged over 5 random seeds per width (a single seed is noisy enough that width alone doesn't
reliably predict sharpness — you need to average across restarts to see the trend clearly),
sharpness should decrease substantially and consistently as width grows: roughly $0.55$ at
width $8$, dropping to about $0.27$ at width $24$, and down to about $0.09$ at width $96$ — more
than a $6\times$ reduction from the narrowest to the widest network tested, while training loss
stays in a broadly similar, comparable range across all three widths (so this isn't simply
"wider networks fit better and therefore look flatter"). This connects two ideas from different
notebooks that aren't obviously the same claim: `04.01`'s "wider networks have friendlier
landscapes" was about *avoiding bad local minima entirely*; this result shows overparameterization
*also* seems to bias which *kind* of minimum you land in even among the good ones, tending toward
flatter basins — consistent with flat minima being one of the mechanisms behind why
heavily overparameterized networks so often generalize well in practice.

</details>

### 🚀 CHALLENGE
6. Vary SAM's `rho` parameter (try $0.01, 0.05, 0.2$) and, for each, train a model and measure
   its sharpness and final training loss. Plot sharpness vs. training loss across the three
   values of `rho` (plus the plain-training baseline) and describe the tradeoff curve you observe.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer reports the tradeoff honestly rather than assuming it's clean at every scale: at
very small `rho` ($0.01$), SAM's ascent step is tiny enough that it barely changes anything
relative to plain training — sharpness and loss both land close to the plain baseline (loss
$\approx0.013$–$0.014$, sharpness $\approx0.27$–$0.29$, statistically indistinguishable given the
sharpness estimator's own noise). The tradeoff becomes clearly visible only once `rho` grows past
this noise floor: at `rho=0.05`, sharpness drops to about $0.13$ (roughly half the baseline) while
training loss rises to about $0.05$ (over $3\times$ the baseline); at `rho=0.2`, sharpness
collapses further to nearly zero ($\approx0.001$) while training loss rises sharply to about
$0.43$ (over $30\times$ the baseline). A strong answer plots sharpness against loss across these
points and notices the relationship isn't linear — the *later* increments of flatness (from
`rho=0.05` to `rho=0.2`) cost proportionally much more training loss than the earlier ones (from
baseline to `rho=0.05`), a classic diminishing-returns tradeoff curve. This connects directly to
why practical SAM usage picks a *moderate* `rho` rather than pushing it as large as possible: past
some point, chasing extra flatness stops being worth the training-loss cost, and a strong answer
notes this optimal point is likely to be task- and architecture-dependent rather than a universal
constant, which is exactly why `rho` is a hyperparameter practitioners tune rather than a fixed
value from theory.

</details>

---

## 📚 Further Reading
- Li et al., ["Visualizing the Loss Landscape of Neural Nets"](https://arxiv.org/abs/1712.09913)
- Foret et al., ["Sharpness-Aware Minimization for Efficiently Improving Generalization"](https://arxiv.org/abs/2010.01412) (SAM)
- Garipov et al., ["Loss Surfaces, Mode Connectivity, and Fast Ensembling of DNNs"](https://arxiv.org/abs/1802.10026)

---

*Next notebook: [Entropy and Information](../05_Information_Theory/01_entropy_and_information.ipynb)*